# 05_split_features: Track 1 split, feature sets and target check
Input: data/processed/itsm_clean.pkl. Outputs: src/project_config.py, src/features.py, data/processed/track1_train.pkl and track1_test.pkl. Aggregates only.
Each %%writefile cell must contain only the file contents; imports go in a separate cell.

## 1. Setup and load the clean data

In [1]:
import os, sys, importlib
import numpy as np
import pandas as pd
sys.path.append("../src")

df = pd.read_pickle("../data/processed/itsm_clean.pkl")

## 2. Split dates in one place (src/project_config.py)

In [2]:
%%writefile ../src/project_config.py
START = "2013-10-01"
TEST_START = "2014-02-01"
CUT = "2014-03-17"
HIGH = [1, 2]

Overwriting ../src/project_config.py


## 3. Build the Track 1 split and add open-time features
Only tickets with a priority label. Train: opened 2013-10-01 to 2014-01-31. Test: 1 February to 16 March 2014. High = priority 1 or 2. Rows are sorted by Open_Time (time-ordered validation depends on it). `month` is deliberately not a feature: the test months never occur in training.

In [3]:
sys.modules.pop("project_config", None)
importlib.invalidate_caches()
from project_config import START, TEST_START, CUT, HIGH

mw = df[(df["Open_Time"] >= START) & (df["Open_Time"] < CUT) & df["Priority"].notna()]
mw = mw.sort_values("Open_Time").reset_index(drop=True)
mw["High"] = mw["Priority"].isin(HIGH).astype(int)
mw["hour"] = mw["Open_Time"].dt.hour
mw["weekday"] = mw["Open_Time"].dt.day_of_week

train = mw[mw["Open_Time"] < TEST_START]
test = mw[mw["Open_Time"] >= TEST_START]
print(len(train), train["High"].sum(), len(test), test["High"].sum())

30845 541 10229 115


## 4. Categorical columns: values unseen in training

In [4]:
cats = ["CI_Name", "CI_Cat", "CI_Subcat", "WBS", "Category"]
for c in cats:
    unseen = (~test[c].isin(train[c].unique())).mean() * 100
    print(f"{c}: {train[c].nunique()} values in train, {unseen:.1f}% of test rows have a value never seen in train")

CI_Name: 2471 values in train, 4.9% of test rows have a value never seen in train
CI_Cat: 13 values in train, 0.0% of test rows have a value never seen in train
CI_Subcat: 58 values in train, 0.0% of test rows have a value never seen in train
WBS: 255 values in train, 0.6% of test rows have a value never seen in train
Category: 4 values in train, 0.0% of test rows have a value never seen in train


## 5. Baseline reference and the saved train and test files

In [5]:
print("Test High rate (a random model's average precision):", round(test["High"].mean(), 4))
print("Accuracy of always predicting not-High:", round(1 - test["High"].mean(), 4))

os.makedirs("../data/processed", exist_ok=True)
train.to_pickle("../data/processed/track1_train.pkl")
test.to_pickle("../data/processed/track1_test.pkl")

Test High rate (a random model's average precision): 0.0112
Accuracy of always predicting not-High: 0.9888


## 6. Do the features carry signal? (aggregates only)

In [6]:
for c in ["Category", "CI_Cat"]:
    g = train.groupby(c)["High"].agg(rows="size", high="sum")
    g["rate_%"] = (g["high"] / g["rows"] * 100).round(2)
    print(g.sort_values("rows", ascending=False).head(8), "\n")

g = train.groupby("CI_Name")["High"].agg(rows="size", high="sum")
big = g["rows"] >= 30
print("CI_Names with at least one High:", int((g["high"] > 0).sum()), "of", len(g))
print("CI_Names with 30+ train rows:", int(big.sum()))
print("Share of train rows on those CIs:", round(g.loc[big, "rows"].sum() / g["rows"].sum() * 100, 1), "%")
print("Share of train High on those CIs:", round(g.loc[big, "high"].sum() / g["high"].sum() * 100, 1), "%")

print("High rate % by hour:", train.groupby("hour")["High"].mean().mul(100).round(1).to_dict())
print("High rate % by weekday:", train.groupby("weekday")["High"].mean().mul(100).round(1).to_dict())

                          rows  high  rate_%
Category                                    
incident                 24416   537    2.20
request for information   6421     4    0.06
complaint                    7     0    0.00
request for change           1     0    0.00 

                    rows  high  rate_%
CI_Cat                                
application        21779   143    0.66
subapplication      5178    23    0.44
computer            2516   345   13.71
storage              424     5    1.18
hardware             255     2    0.78
software             200     1    0.50
displaydevice        143     0    0.00
officeelectronics    106     0    0.00 

CI_Names with at least one High: 358 of 2471
CI_Names with 30+ train rows: 139
Share of train rows on those CIs: 83.5 %
Share of train High on those CIs: 23.7 %
High rate % by hour: {0: 2.3, 1: 0.0, 2: 0.0, 3: 0.0, 4: 33.3, 5: 23.8, 6: 24.3, 7: 5.4, 8: 1.9, 9: 2.0, 10: 1.6, 11: 1.6, 12: 1.7, 13: 1.6, 14: 1.4, 15: 1.0, 16: 1.7, 17: 1.9

### Findings
- Category: incidents have a 2.20% High rate (537 of 24,416), requests for information 0.06% (4 of 6,421).
- CI_Cat: computer tickets have a 13.71% High rate (345 of 2,516, which is 64% of all High tickets in training), against 0% to 1.18% for the other asset types.
- CI_Name: 139 of 2,471 CIs have 30 or more training rows. They hold 83.5% of tickets but only 23.7% of High tickets, so most High tickets sit on rare CIs.
- Hour and weekday: the High rate is 23.8% to 33.3% at hours 4 to 6 and 8.8% (Saturday) and 19.4% (Sunday) at weekends, against 1.4% to 2.1% on weekdays (some hours have small counts).

## 7. Feature sets (src/features.py)

In [7]:
%%writefile ../src/features.py
asset_cols = ["CI_Cat", "CI_Subcat", "WBS", "Category"]
ci_name    = ["CI_Name"]
time_feats = ["hour", "weekday"]
matrix_cols = ["Impact", "Urgency"]

post_event = ["Status", "Resolved_Time", "Close_Time", "Reopen_Time", "Has_Reopen_Time",
              "Handle_Time_hrs", "Elapsed_hrs", "Closure_Code"]
unsure_timing = ["No_of_Reassignments", "No_of_Related_Interactions", "No_of_Related_Incidents",
                 "No_of_Related_Changes", "Has_Related_Change", "KB_number"]
identifiers = ["Incident_ID", "Related_Interaction", "Related_Change"]

set_a = asset_cols + time_feats + matrix_cols   # includes Impact and Urgency
set_b = asset_cols + time_feats                 # the honest test

Overwriting ../src/features.py


## 8. Check that every column is classified

In [8]:
sys.modules.pop("features", None)
importlib.invalidate_caches()
from features import *

known = set(asset_cols + ci_name + time_feats + matrix_cols + post_event
            + unsure_timing + identifiers + ["Priority", "High", "Open_Time"])
print(sorted(set(train.columns) - known))

[]


## 9. Explicit X and y (Set B features)

In [9]:
X_train, y_train = train[set_b].astype(str), train["High"]
X_test,  y_test  = test[set_b].astype(str),  test["High"]
print(X_train.shape, X_test.shape, int(y_train.sum()), int(y_test.sum()))

assert list(X_train.columns) == list(X_test.columns) == set_b
assert not set(X_train.columns) & set(post_event + unsure_timing + identifiers + matrix_cols + ci_name)
assert set(y_train.unique()) <= {0, 1} and set(y_test.unique()) <= {0, 1}
print("features:", list(X_train.columns))

(30845, 6) (10229, 6) 541 115
features: ['CI_Cat', 'CI_Subcat', 'WBS', 'Category', 'hour', 'weekday']


## 10. Target check: High against Priority and Impact

In [10]:
print(pd.crosstab(train["High"], train["Priority"]))
print(y_train.value_counts().to_dict(), y_test.value_counts().to_dict())
print(pd.crosstab(mw["Impact"], mw["High"]))

Priority  2.0   3.0    4.0    5.0
High                             
0           0  3517  15180  11607
1         541     0      0      0
{0: 30304, 1: 541} {0: 10114, 1: 115}
High        0    1
Impact            
1.0         0    3
2.0         0  649
3.0      4679    4
4.0     20378    0
5.0     15361    0


### Findings
- High is priority 1 or 2. In the training period every High ticket is priority 2 (541); the 3 priority-1 tickets in the window all fall in the test period.
- High is almost "Impact 1 or 2": of the 656 High tickets in the modelling window, 652 have Impact 1 or 2 (3 + 649) and 4 have Impact 3. Priority is a lookup of Impact and Urgency, which is why Impact and Urgency are excluded from the headline features.